In [ ]:
"""
Copyright (c) 2021-2024 D-Robotics Corporation

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
"""

In [ ]:
!cat /proc/meminfo | grep Mem

In [ ]:
# 导入所需要的包
import numpy as np
import cv2
import time
import os
import argparse
import hbm_runtime
from typing import Dict, Optional

# For displaying images in Jupyter
import matplotlib.pyplot as plt

# ============== 工具函数 (从utils中提取) ==============

def bgr_to_nv12_planes(image: np.ndarray) -> tuple:
    """
    将BGR图像转换为NV12格式（Y和UV平面）。
    """
    height, width = image.shape[:2]
    area = height * width

    # 转换为平面YUV I420格式
    yuv420p = cv2.cvtColor(image, cv2.COLOR_BGR2YUV_I420)
    yuv420p = yuv420p.reshape((area * 3 // 2,))

    # 提取Y、U、V平面
    y = yuv420p[:area].reshape((height, width))
    u = yuv420p[area:area + area // 4].reshape((height // 2, width // 2))
    v = yuv420p[area + area // 4:].reshape((height // 2, width // 2))

    # 交错U和V形成UV平面
    uv = np.stack((u, v), axis=-1)

    # 添加批次和通道维度
    y = y[np.newaxis, :, :, np.newaxis]
    uv = uv[np.newaxis, :, :, :]

    return y, uv


def resized_image(img: np.ndarray, input_W: int, input_H: int,
                  resize_type: int = 1,
                  interpolation=cv2.INTER_NEAREST) -> np.ndarray:
    """
    使用直接调整大小或letterbox策略调整图像大小。
    """
    img_h, img_w = img.shape[:2]

    if resize_type == 0:  # 直接调整大小
        resized = cv2.resize(img, (input_W, input_H), interpolation=interpolation)
    elif resize_type == 1:  # Letterbox调整大小（保持长宽比）
        scale = min(input_H / img_h, input_W / img_w)
        new_w, new_h = int(img_w * scale), int(img_h * scale)
        resized = cv2.resize(img, (new_w, new_h))

        pad_w = input_W - new_w
        pad_h = input_H - new_h
        left, right = pad_w // 2, pad_w - pad_w // 2
        top, bottom = pad_h // 2, pad_h - pad_h // 2

        # 用灰色填充图像 (127,127,127)
        resized = cv2.copyMakeBorder(resized, top, bottom, left, right,
                                     borderType=cv2.BORDER_CONSTANT,
                                     value=(127, 127, 127))
    else:
        raise ValueError(f"无效的resize_type: {resize_type}，必须是0或1")

    return resized


def print_top1_prediction(output: np.ndarray, idx2label: dict = None) -> tuple:
    """
    打印top-1分类预测结果并返回预测信息。
    """
    # 带稳定性调整的Softmax
    exp_logits = np.exp(output - np.max(output))
    probabilities = exp_logits / np.sum(exp_logits)

    # Top-1索引和概率
    top_idx = np.argmax(probabilities)
    top_prob = probabilities[top_idx]
    
    # 获取标签名称
    label = idx2label[top_idx] if idx2label and top_idx in idx2label else f"类别 {top_idx}"
    
    print(f"Top-1 预测结果: {label} (置信度: {top_prob:.4f})")
    
    return top_idx, top_prob, label


def display_image(image_cv, title="Image", size=(6, 6)):
    """使用Matplotlib显示OpenCV图像（BGR转RGB）。"""
    # 将BGR转换为RGB以供Matplotlib使用
    image_rgb = cv2.cvtColor(image_cv, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=size)
    plt.imshow(image_rgb)
    plt.title(title)
    plt.axis('off') # 隐藏坐标轴
    plt.show()

# ============== 工具函数结束 ==============

# ResNet-18 模型推理演示

本notebook演示了如何进行ResNet-18模型的图像分类推理。

## 1. 模型和数据验证

首先检查模型文件和测试图像是否在预期位置可用。

In [ ]:
# --- 配置参数 ---
NOTEBOOK_DIR = '.' 

MODEL_NAME = 'resnet18_224x224_nv12.hbm'
MODEL_PATH = os.path.join(NOTEBOOK_DIR, './model', MODEL_NAME)

IMAGE_NAME = 'zebra_cls.jpg'
IMAGE_PATH = os.path.join(NOTEBOOK_DIR, './data', IMAGE_NAME)

LABEL_FILE = '/app/res/labels/imagenet1000_clsidx_to_labels.txt'

# --- 配置参数结束 ---

print(f"Notebook目录: {os.path.abspath(NOTEBOOK_DIR)}")
print(f"模型路径: {os.path.abspath(MODEL_PATH)}")
print(f"图像路径: {os.path.abspath(IMAGE_PATH)}")
print(f"标签文件路径: {LABEL_FILE}")

# 检查文件是否存在
if not os.path.exists(MODEL_PATH):
    print(f"警告: 在 {MODEL_PATH} 未找到模型文件")
if not os.path.exists(IMAGE_PATH):
    print(f"警告: 在 {IMAGE_PATH} 未找到图像文件")
if not os.path.exists(LABEL_FILE):
    print(f"警告: 在 {LABEL_FILE} 未找到标签文件")

# 加载标签映射
idx2label = None
if os.path.exists(LABEL_FILE):
    try:
        with open(LABEL_FILE, "r") as f:
            idx2label = eval(f.read())
        print(f"成功加载 {len(idx2label)} 个类别标签")
    except Exception as e:
        print(f"加载标签文件失败: {e}")
        idx2label = None
else:
    print("未找到标签文件，将使用类别ID")

## 2. 模型类定义

定义ResNet-18封装类，该类包含模型加载、预处理、推理和后处理的所有步骤。

In [ ]:
# ResNet18 分类模型
class ResNet18:
    """
    @brief 使用HB_HBMRuntime运行ResNet-18模型推理的封装类。
    """

    def __init__(self, model_path):
        """
        @brief 使用模型路径初始化ResNet-18模型并提取输入/输出详细信息。
        """
        # 加载模型运行时
        self.model = hbm_runtime.HB_HBMRuntime(model_path)

        # 获取模型名称和输入/输出名称
        self.model_name = self.model.model_names[0]
        self.input_names = self.model.input_names[self.model_name]
        self.output_names = self.model.output_names[self.model_name]
        self.shapes = self.model.input_shapes[self.model_name]

        # 提取输入分辨率（高度，宽度）
        self.input_H = self.shapes[self.input_names[0]][1]
        self.input_W = self.shapes[self.input_names[0]][2]
        
        print(f"模型已加载: {self.model_name}")
        print(f"输入尺寸: {self.input_W} x {self.input_H}")
        print(f"输入名称: {self.input_names}")
        print(f"输出名称: {self.output_names}")

    def pre_process(self, img: np.ndarray, resize_type: int = 1):
        """
        @brief 预处理输入图像以匹配模型输入格式。
        """
        # 调整大小并转换图像为NV12格式
        resize_img = resized_image(img, self.input_W, self.input_H, resize_type)
        y, uv = bgr_to_nv12_planes(resize_img)

        return {
            self.model_name: {
                self.input_names[0]: y,
                self.input_names[1]: uv
            }
        }

    def forward(self, input_tensor):
        """
        @brief 使用预处理的输入张量运行前向推理。
        """
        outputs = self.model.run(input_tensor)
        return outputs[self.model_name]

    def post_process(self, outputs, idx2label=None):
        """
        @brief 后处理输出并返回top-1预测结果。
        """
        # 获取并显示top-1预测结果
        top_idx, top_prob, label = print_top1_prediction(outputs[self.output_names[0]][0], idx2label)
        return outputs[self.output_names[0]][0], top_idx, top_prob, label

print("ResNet-18模型类定义成功")

## 3. 模型加载和初始化

加载HBM模型文件并初始化ResNet-18推理管道。此步骤将验证模型结构并提取输入/输出张量信息。

In [ ]:
print(f"加载模型: {MODEL_PATH}")

try:
    # 初始化ResNet-18模型
    resnet18 = ResNet18(MODEL_PATH)
    print("模型初始化成功!")
    
    # 打印详细的模型信息
    print(f"\n=== 模型详细信息 ===")
    print(f"模型数量: {resnet18.model.model_count}")
    print(f"输入张量形状: {resnet18.model.input_shapes}")
    print(f"输出张量形状: {resnet18.model.output_shapes}")
    
except Exception as e:
    print(f"加载模型时出错: {e}")
    print("请确保运行环境可用且模型路径正确。")
    resnet18 = None

## 4. 图像加载和预处理

加载测试图像并应用必要的预处理步骤：
- 调整图像大小以匹配模型输入要求（224x224）
- 将BGR格式转换为NV12格式进行推理
- 显示原始图像和预处理后的图像以进行验证

In [ ]:
print(f"从以下路径加载图像: {IMAGE_PATH}")
original_bgr_image = cv2.imread(IMAGE_PATH)

if original_bgr_image is None:
    print(f"在 {IMAGE_PATH} 加载图像失败。请检查路径。")
    preprocessed_input = None
else:
    print(f"原始图像已加载。形状: {original_bgr_image.shape}")
    display_image(original_bgr_image, title="Original Image")

    if resnet18 is not None:
        print(f"为模型输入({resnet18.input_W}x{resnet18.input_H})预处理图像...")
        
        # 使用模型的预处理方法
        preprocessed_input = resnet18.pre_process(original_bgr_image)
        
        # 显示调整大小后的图像以进行视觉验证
        resized_img = resized_image(original_bgr_image, resnet18.input_W, resnet18.input_H)
        display_image(resized_img, title=f"Resized Image ({resnet18.input_W}x{resnet18.input_H})")
        
        print("图像预处理成功完成!")
        print(f"预处理输入结构: {list(preprocessed_input.keys())}")
    else:
        print("跳过预处理 - 模型未加载")
        preprocessed_input = None

## 5. 模型推理

使用预处理后的输入通过ResNet-18模型进行前向推理。此步骤测量推理时间并验证输出张量形状。

In [ ]:
if resnet18 is not None and preprocessed_input is not None:
    print("运行模型推理...")
    start_time_inference = time.time()
    try:
        # 使用模型的前向方法运行推理
        outputs = resnet18.forward(preprocessed_input)
        inference_time = time.time() - start_time_inference
        print(f"推理在 {inference_time:.4f} 秒内完成。")
        print(f"输出形状: {[(name, arr.shape) for name, arr in outputs.items()]}")
        
        # 存储推理结果用于后处理
        inference_successful = True
    except Exception as e:
        print(f"推理过程中出错: {e}")
        outputs = None
        inference_successful = False
else:
    print("跳过推理 - 模型或预处理输入不可用。")
    outputs = None
    inference_successful = False

## 6. 后处理和结果

处理模型输出以生成分类预测：
- 应用softmax将logits转换为概率
- 提取置信度最高的Top-K预测结果
- 准备结果以进行可视化

In [ ]:
if inference_successful and outputs is not None:
    print("\n开始后处理...")
    t0 = time.time()
    
    # 使用模型内置的后处理方法
    try:
        classification_output, top_idx, top_prob, top_label = resnet18.post_process(outputs, idx2label)
        t1 = time.time()
        print(f"后处理在 {(t1 - t0):.4f} 秒内完成")
        
        # 存储预测结果
        prediction_results = {
            'idx': top_idx,
            'prob': top_prob,
            'label': top_label
        }
        
    except Exception as e:
        print(f"后处理过程中出错: {e}")
        classification_output = None
        prediction_results = None
        
else:
    print("跳过后处理 - 推理未成功。")
    classification_output = None
    prediction_results = None

## 7. 结果可视化

显示最终分类结果：
- 按置信度排序的Top-5预测结果
- 标注了最高预测结果的原始图像
- 完整的推理管道摘要

In [ ]:
if prediction_results is not None and original_bgr_image is not None:
    print("\n" + "=" * 10, "最终结果显示", "=" * 10)
    
    # 获取预测结果
    top_idx = prediction_results['idx']
    top_prob = prediction_results['prob']
    top_label = prediction_results['label']
    
    print(f"最终预测结果: {top_label} (类别ID: {top_idx}, 置信度: {top_prob:.4f})")
    
    # 创建标注图像
    annotated_image = original_bgr_image.copy()
    
    # 在图像上添加预测结果文本
    font = cv2.FONT_HERSHEY_SIMPLEX
    
    # 准备显示文本
    # 如果标签太长，需要换行或截断
    max_length = 40  # 最大字符长度
    if len(top_label) > max_length:
        # 尝试在合适位置断行
        words = top_label.split(' ')
        line1 = ""
        line2 = ""
        for word in words:
            if len(line1 + word) < max_length:
                line1 += word + " "
            else:
                line2 += word + " "
        display_lines = [line1.strip(), line2.strip()]
    else:
        display_lines = [top_label]
    
    # 显示类别名称（主要信息）
    y_pos = 40
    for i, line in enumerate(display_lines):
        if line:  # 确保不是空行
            cv2.putText(annotated_image, line, (10, y_pos + i * 35),
                        font, 0.8, (0, 255, 0), 2, cv2.LINE_AA)
    
    # 显示置信度
    conf_text = f"Confidence: {top_prob:.3f}"
    cv2.putText(annotated_image, conf_text, (10, y_pos + len(display_lines) * 35 + 10),
                font, 0.7, (255, 255, 0), 2, cv2.LINE_AA)
    
    # 显示模型名称
    model_text = "ResNet-18"
    cv2.putText(annotated_image, model_text, (10, y_pos + len(display_lines) * 35 + 45),
                font, 0.6, (255, 0, 0), 2, cv2.LINE_AA)
    
    display_image(annotated_image, title="Classification Result")
    
else:
    print("无结果可显示 - 分类未成功。")

print("\n--- ResNet-18 推理完成 ---")

In [ ]:
!hrt_model_exec perf --model_file ./model/resnet18_224x224_nv12.hbm \
                    --core_id=0 \
                    --frame_count=200 \
                    --perf_time=0 \
                    --thread_num=3